# ⚓ Módulo 8: Big Data Analytics (PySpark)
### Maestría en Logística y Transporte Marítimo — UMIP
#### Preparado por: Jcenteno

Procesamiento masivo de señales AIS globales usando el motor Spark en la nube.

---

In [ ]:
# 1. Instalación de Java y Spark en Colab (Solo ejecutar una vez)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Logistica_UMIP").getOrCreate()
print("⚡ ¡Spark activo y listo!")

## 1. Carga Masiva y Filtrado Geográfico
Spark permite trabajar con archivos que no caben en memoria (RAM).

In [ ]:
# Simulamos lectura de archivo grande
try:
    # Intentar cargar desde GitHub (Colab) o Local
    url = "https://jacenteno.github.io/Analisis_Datos_Maritimo/02_Datasets/ais_muestra_canal.csv"
    try:
        df = spark.read.csv(url)
        print("✅ Datos cargados desde GitHub")
    except:
        df = spark.read.csv("../02_Datasets/ais_muestra_canal.csv")
        print("🏠 Datos cargados localmente")
except:
    print("📂 Creando DataFrame de prueba en Spark...")
    data = [(355789, 8.9, -79.6, "Container"), (371234, 7.5, -78.0, "Tanker")] * 50000
    df_ais = spark.createDataFrame(data, ["mmsi", "lat", "lon", "type"])

# Filtrado de Zona Panamá
zona_panama = df_ais.filter((F.col("lat") < 9.5) & (F.col("lat") > 7.0))
print(f"Registros en Panamá: {zona_panama.count()}")

## 2. Agregación Big Data
Calculamos estadísticas de flota de forma distribuida.

In [ ]:
stats = zona_panama.groupBy("type").agg(
    F.count("mmsi").alias("conteo"),
    F.countDistinct("mmsi").alias("buques_unicos")
)

stats.show()

## 🚀 Laboratorio M8
1. **Tarea**: Identifica el Top 3 de tipos de buques con mayor presencia en el Canal este mes.
2. **Exportar**: Convierte los resultados de Spark a Pandas (`toPandas()`) y expórtalos como un reporte CSV compacto.